<a href="https://colab.research.google.com/github/muajnstu/Portfolio-Management-using-RL/blob/master/Taiwan_Implementation_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taiwan 77-stock DJIA core workflow: five independent seeds

**Seeds 42, 123, 456, 789, 2026.** Reuse the *completed original* seed-42 CNN, five RL experts, four gates, and three architecture routing tests. The four additional seeds **train fresh CNN, experts and gates** in isolated Drive folders and evaluate CNN2D, GRU and LSTM dense, Top-3, Top-2, Top-1 and validation-selected routes. Fresh seeds never load seed-42 model checkpoints. They share only the dated 77-stock universe, raw/prepared inputs, frozen HMM and samples. The original active-DIR reward, portfolio environment, costs, free-mode training budget, and 2004–2025 chronology remain unchanged.

This is the original screened **77-stock Taiwan sample**, not historical TW50 membership. Benchmark results (Buy and Hold, Equal Weight and MVO) are a fixed reference copied from seed 42. This notebook does not introduce a Taiwan index benchmark or a No-HMM branch. It uses 2015 validation to pick each route and cross-architecture winner; 2016–2025 is scored out of sample. Run top to bottom in Colab with the original project available in the mounted Google Drive. Resume by running from the top again after a disconnect. Training four new pipelines can require substantial time and storage.


In [ ]:
%pip -q install 'stable-baselines3>=2.4,<3' 'gymnasium>=1.0,<1.3' 'hmmlearn>=0.3,<1' 'pandas-datareader>=0.10' 'scikit-learn>=1.3' 'joblib>=1.3' 'torch>=2'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 17.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,sys,json,io,base64,zipfile,hashlib,shutil,subprocess,importlib.util
import numpy as np,pandas as pd,torch
from IPython.display import display
REFERENCE=Path('/content/drive/MyDrive/tw50_2004_2025_alpha_dynmoe/continuous_frozen_2016_2025')
COHORT=REFERENCE.parent/'five_seed_continuous_2016_2025'
CODE=Path('/content/tw50_five_seed_source')
SEEDS=(42,123,456,789,2026)
# Original seed-42 gate is 2026. Last new run uses 2027 to keep its gate fresh.
GATE_SEEDS={42:2026,123:123,456:456,789:789,2026:2027}
COHORT.mkdir(parents=True,exist_ok=True);CODE.mkdir(parents=True,exist_ok=True)
assert (REFERENCE/'data/raw/selected_universe.json').exists(),REFERENCE
assert (REFERENCE/'results/active_dir_reward/neural_gate_study/test_metrics.csv').exists(), 'Complete the seed-42 study first'
assert len(json.loads((REFERENCE/'data/raw/selected_universe.json').read_text())['selected_tickers'])==77
print('Reuse only seed 42:',REFERENCE)
print('Train fresh seeds in:',COHORT)


Mounted at /content/drive
Reuse only seed 42: /content/drive/MyDrive/tw50_2004_2025_alpha_dynmoe/continuous_frozen_2016_2025
Train fresh seeds in: /content/drive/MyDrive/tw50_2004_2025_alpha_dynmoe/five_seed_continuous_2016_2025


In [ ]:
BUNDLE_SHA256='98ce52290d5c02663ad972e0783363e618e09c0de3f2193220475be6358d4baa'
BUNDLE_B64="""..."""
payload=base64.b64decode(BUNDLE_B64)
assert hashlib.sha256(payload).hexdigest()==BUNDLE_SHA256
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    assert all(not p.startswith('/') and '..' not in Path(p).parts for p in z.namelist())
    z.extractall(CODE)
assert (CODE/'phase2_9_gru_lstm_core.py').exists()
print('Verified embedded Taiwan source bundle:',BUNDLE_SHA256)


Verified embedded Taiwan source bundle: 98ce52290d5c02663ad972e0783363e618e09c0de3f2193220475be6358d4baa


In [ ]:
meta_gate_path = CODE / 'meta_gate.py'
with open(meta_gate_path, 'r') as f:
    content = f.read()

# Remove all occurrences of 'import os' to ensure a clean slate before re-inserting.
lines = content.splitlines(keepends=True)
lines_without_import_os = [line for line in lines if 'import os\n' not in line]
content_cleaned = ''.join(lines_without_import_os)

# Find the line with 'from __future__ import annotations'
future_import_index = -1
lines_cleaned = content_cleaned.splitlines(keepends=True)
for i, line in enumerate(lines_cleaned):
    if 'from __future__ import annotations' in line:
        future_import_index = i
        break

# Prepare the 'import os' line
import_os_line = 'import os\n'

# Re-insert 'import os' at the correct position
if future_import_index != -1:
    # Insert 'import os' after the future import
    lines_cleaned.insert(future_import_index + 1, import_os_line)
    new_content = ''.join(lines_cleaned)
    print(f"Moved 'import os' after __future__ import in {meta_gate_path}")
else:
    # If no future import, add 'import os' at the very beginning
    new_content = import_os_line + content_cleaned
    print(f"Added 'import os' to the beginning of {meta_gate_path} (no __future__ import found)")

with open(meta_gate_path, 'w') as f:
    f.write(new_content)

print('meta_gate.py modified successfully.')

Moved 'import os' after __future__ import in /content/tw50_five_seed_source/meta_gate.py
meta_gate.py modified successfully.


In [ ]:
MODE='free'  # Exactly the original completed run's training budget and reward.
ARCHS=('CNN2D','GRU','LSTM')
AGENTS=('DDPG','TD3','SAC','PPO','A2C')
def root_for(seed):return REFERENCE if seed==42 else COHORT/f'seed_{seed}'
def env_for(root,seed,arch=None):
    e=os.environ.copy()
    e.update(FIXED29_PROJECT_ROOT=str(root),FIXED29_USE_HMM='1',
             FIXED29_EXTENSION_MODE=MODE,FIXED29_EXTENSION_OVERWRITE='0',
             TW50_SEED=str(seed),TW50_GATE_SEED=str(GATE_SEEDS[seed]),
             FIXED29_EXTENSION_SEED=str(GATE_SEEDS[seed]),PYTHONPATH=str(CODE))
    if arch:
        e.update(FIXED29_PRIMARY_GATE=arch,
                 FIXED29_EXTENSION_ROOT=str(root/'route_comparison'/arch.lower()))
    return e
def execute(root,seed,stage,args,arch=None):
    logdir=root/'five_seed_logs';logdir.mkdir(parents=True,exist_ok=True)
    log=logdir/f'{stage}.log';print('RUNNING',root.name,stage,'log:',log,flush=True)
    with log.open('w') as f:
        p=subprocess.Popen([sys.executable,'-u',str(CODE/args[0]),*args[1:]],
            cwd=CODE,env=env_for(root,seed,arch),stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1)
        tail=[]
        for line in p.stdout:
            f.write(line);f.flush();tail=(tail+[line.rstrip()])[-80:]
            if any(tag in line for tag in ('Training ','epoch=','PHASE 4','OOS GATES','COMPLETE','reused')):
                print(line.rstrip(),flush=True)
        code=p.wait()
    if code:raise RuntimeError(f'{stage} failed; {log}\n'+'\n'.join(tail))
def stage_fixed_inputs(root):
    for part in ('raw','features','prepared','samples'):
        src=REFERENCE/'data'/part;dest=root/'data'/part
        if not src.exists():raise FileNotFoundError(src)
        shutil.copytree(src,dest,dirs_exist_ok=True)
    assert (root/'data/samples/train.npz').exists()
    assert (root/'data/prepared/hmm_train_only.joblib').exists()
    a=json.loads((REFERENCE/'data/raw/selected_universe.json').read_text())['selected_tickers']
    b=json.loads((root/'data/raw/selected_universe.json').read_text())['selected_tickers']
    assert a==b and len(b)==77
def verify_cnn(root,seed):
    p=root/'models/shared/cnn/cnn_best.pt'
    if not p.exists():return False
    ck=torch.load(p,map_location='cpu',weights_only=False)
    if int(ck['seed'])!=seed or not bool(ck['use_hmm']):raise ValueError(f'CNN checkpoint mismatch: {p}')
    return True
def verify_experts(root,seed):
    base=root/'models/active_dir_agents/agents'
    mf=base/'manifest.json'
    if not mf.exists():return False
    rows=json.loads(mf.read_text()).get('models',[])
    for agent in AGENTS:
        matches=[x for x in rows if x.get('agent')==agent and x.get('run_mode')=='free_colab' and int(x.get('seed',-1))==seed]
        folder=base/'free_colab'/agent.lower()/f'seed_{seed}'
        if len(matches)!=1 or not (folder/'best_model.zip').exists() or not (folder/'summary.json').exists():return False
        if Path(matches[0]['model_path']).resolve()!= (folder/'best_model.zip').resolve():
            raise RuntimeError(f'Manifest points outside this seed: {agent} {root}')
    return True
def verify_gates(root):
    base=root/'models/active_dir_agents/neural_gate_candidates'
    return (base/'selected_gate.json').exists() and all(
        (base/a/'model.pt').exists() and (base/a/'scaler.joblib').exists()
        for a in ('cnn2d','gru','lstm','mlp_32x32'))
def routes_for(seed,arch):
    if seed==42:
        return ((REFERENCE/'phase2_9_extension/phase4_routing') if arch=='CNN2D' else
                 (REFERENCE/'results/gru_lstm_gate_comparison_2016_2025'/arch.lower()/'phase4_routing'))
    return root_for(seed)/'route_comparison'/arch.lower()/'phase4_routing'
def complete_route(folder):
    return all((folder/n).exists() for n in ('validation_routing_selection.csv','test_routing_metrics.csv')) and all(
        (folder/'test'/f'route_{i:02d}'/'daily_returns.csv').exists() for i in range(4))
def run_seed(seed):
    root=root_for(seed)
    if seed!=42:
        root.mkdir(parents=True,exist_ok=True)
        recipe={'seed':seed,'gate_seed':GATE_SEEDS[seed],'source_sha256':BUNDLE_SHA256,
                'source_reference':str(REFERENCE),'mode':'free','reward':'original_active_DIR'}
        recipe_file=root/'five_seed_recipe.json'
        if recipe_file.exists():assert json.loads(recipe_file.read_text())==recipe,'Configuration changed; use a new folder'
        else:recipe_file.write_text(json.dumps(recipe,indent=2))
        stage_fixed_inputs(root)
        if not verify_cnn(root,seed):execute(root,seed,'05_cnn',['05_train_cnn.py','--seed',str(seed),'--quick'])
        assert verify_cnn(root,seed)
        states=root/'data/states'
        if not all((states/f'{x}.npz').exists() for x in ('train','validation','test')):
            execute(root,seed,'06_states',['06_precompute_states.py'])
        if not verify_experts(root,seed):execute(root,seed,'08_agents',['08_train_agents.py','--mode',MODE,'--agent','all'])
        assert verify_experts(root,seed),'RL stage incomplete'
        if not verify_gates(root):execute(root,seed,'09_gate_training',['09_neural_gate_study.py','--mode',MODE,'--train-select'])
        assert verify_gates(root),'Gate stage incomplete'
        if not (root/'results/active_dir_reward/neural_gate_study/test_metrics.csv').exists():
            execute(root,seed,'09_gate_test',['09_neural_gate_study.py','--mode',MODE,'--evaluate'])
    else:
        assert verify_cnn(root,42) and verify_experts(root,42) and verify_gates(root), 'Original seed 42 is incomplete'
    for arch in ARCHS:
        folder=routes_for(seed,arch)
        if not complete_route(folder):
            if seed==42:raise FileNotFoundError(f'Complete original seed-42 {arch} route first: {folder}')
            execute(root,seed,'route_'+arch.lower(),['phase2_9_gru_lstm_core.py','--phase','4'],arch)
        assert complete_route(folder),folder
    print('COMPLETE',seed,root,flush=True)
for seed in SEEDS:run_seed(seed)

COMPLETE 42 /content/drive/MyDrive/tw50_2004_2025_alpha_dynmoe/continuous_frozen_2016_2025
COMPLETE 123 /content/drive/MyDrive/tw50_2004_2025_alpha_dynmoe/five_seed_continuous_2016_2025/seed_123
COMPLETE 456 /content/drive/MyDrive/tw50_2004_2025_alpha_dynmoe/five_seed_continuous_2016_2025/seed_456
COMPLETE 789 /content/drive/MyDrive/tw50_2004_2025_alpha_dynmoe/five_seed_continuous_2016_2025/seed_789
RUNNING seed_2026 08_agents log: /content/drive/MyDrive/tw50_2004_2025_alpha_dynmoe/five_seed_continuous_2016_2025/seed_2026/five_seed_logs/08_agents.log
Training DDPG | seed=2026 | timesteps=20000 | device=cpu | calibration_offsets=(0, 5, 9)
Training TD3 | seed=2026 | timesteps=20000 | device=cpu | calibration_offsets=(0, 5, 9)
Training SAC | seed=2026 | timesteps=20000 | device=cpu | calibration_offsets=(0, 5, 9)
Training PPO | seed=2026 | timesteps=20000 | device=cpu | calibration_offsets=(0, 5, 9)
Training A2C | seed=2026 | timesteps=20000 | device=cpu | calibration_offsets=(0, 5, 9)
RU

In [ ]:
# Execute after all five complete. Report matched strategies and validation-selected routes.
out=COHORT/'summary';out.mkdir(parents=True,exist_ok=True)
metrics=('annualized_return','sharpe_ratio','sortino_ratio','maximum_drawdown',
         'annualized_volatility','final_portfolio_value')
models=[];routes=[];cross=[]
for seed in SEEDS:
    root=root_for(seed)
    p=root/'results/active_dir_reward/neural_gate_study/test_metrics.csv'
    if not p.exists():raise FileNotFoundError(p)
    m=pd.read_csv(p);m['Seed']=seed;models.append(m)
    selected=m.loc[m.Is_Selected_Proposed.fillna(False).astype(bool)].copy()
    if len(selected)!=1:raise RuntimeError(f'Expected one pretest gate winner: {p}')
    selected['Strategy']='Pretest Selected Gate';models.append(selected)
    vf=[]
    for arch in ARCHS:
        folder=routes_for(seed,arch)
        if not complete_route(folder):raise FileNotFoundError(folder)
        v=pd.read_csv(folder/'validation_routing_selection.csv');v['Architecture']=arch;vf.append(v)
        t=pd.read_csv(folder/'test_routing_metrics.csv');t['Seed']=seed;t['Architecture']=arch
        routes.append(t)
        chosen=t.loc[t.Selected_PreOOS.fillna(False).astype(bool)].copy()
        if len(chosen)!=1:raise RuntimeError(f'Expected one pretest route: {folder}')
        chosen['Spec']='Pretest Selected Route';routes.append(chosen)
    v=pd.concat(vf,ignore_index=True)
    for name,asc in [('annualized_return',False),('sharpe_ratio',False),('sortino_ratio',False),('maximum_drawdown',True)]:
        v['Rank_'+name]=pd.to_numeric(v[name],errors='coerce').rank(ascending=asc,method='average',na_option='bottom')
    v['Cross_Arch_Mean_Rank']=v[['Rank_'+x for x in ('annualized_return','sharpe_ratio','sortino_ratio','maximum_drawdown')]].mean(axis=1)
    v=v.sort_values(['Cross_Arch_Mean_Rank','Rank_sharpe_ratio','Rank_annualized_return']).reset_index(drop=True)
    v.insert(0,'Seed',seed);cross.append(v)
    winner=v.iloc[0]
    matching=[x for x in routes if len(x) and x.Seed.eq(seed).all() and x.Architecture.eq(winner.Architecture).all() and x.Spec.eq(winner.Spec).any()]
    candidates=[x.loc[x.Spec.eq(winner.Spec)].copy() for x in matching if x.Spec.eq(winner.Spec).any()]
    if len(candidates)!=1 or len(candidates[0])!=1:raise RuntimeError(f'Cross architecture winner missing for seed {seed}')
    candidates[0]['Chosen_Architecture']=candidates[0]['Architecture']
    candidates[0]['Architecture']='VALIDATION_SELECTED'
    candidates[0]['Spec']='Pretest Selected Architecture and Route';routes.append(candidates[0])
model=pd.concat(models,ignore_index=True);route=pd.concat(routes,ignore_index=True)
model.to_csv(out/'individual_gate_expert_tests.csv',index=False)
route.to_csv(out/'individual_route_tests.csv',index=False)
pd.concat(cross,ignore_index=True).to_csv(out/'validation_cross_architecture_rankings.csv',index=False)
def five_seed_summary(frame,group):
    cols=[c for c in metrics if c in frame]
    long=frame.melt(id_vars=['Seed',*group],value_vars=cols,var_name='Metric',value_name='Value')
    long['Value']=pd.to_numeric(long.Value,errors='coerce')
    summary=long.groupby([*group,'Metric'],dropna=False).Value.agg(
        N='count',Mean='mean',SD='std',Median='median',Min='min',Max='max').reset_index()
    return summary.loc[summary.N.eq(5)].copy()
g=five_seed_summary(model,['Strategy']);r=five_seed_summary(route,['Architecture','Spec'])
required=['Pretest Selected Gate',*[f'Full-Control Expert — {a}' for a in AGENTS]]
for label in required:
    for metric in ('annualized_return','sharpe_ratio','maximum_drawdown'):
        if len(g.loc[g.Strategy.eq(label)&g.Metric.eq(metric)])!=1:raise RuntimeError(f'Five seeds incomplete: {label} {metric}')
for arch in ARCHS:
    for spec in ('Pretest Selected Route',*('{'+'"mode": "'+m+'"}' for m in ('dense','top3','top2','top1'))):
        for metric in ('annualized_return','sharpe_ratio','maximum_drawdown'):
            if len(r.loc[r.Architecture.eq(arch)&r.Spec.eq(spec)&r.Metric.eq(metric)])!=1:
                raise RuntimeError(f'Five route seeds incomplete: {arch} {spec} {metric}')
for metric in ('annualized_return','sharpe_ratio','maximum_drawdown'):
    if len(r.loc[r.Architecture.eq('VALIDATION_SELECTED') &
                 r.Spec.eq('Pretest Selected Architecture and Route') & r.Metric.eq(metric)])!=1:
        raise RuntimeError(f'Five cross-architecture seeds incomplete: {metric}')
g.to_csv(out/'five_seed_gate_expert_summary.csv',index=False)
r.to_csv(out/'five_seed_route_summary.csv',index=False)
baseline=REFERENCE/'results/taiwan_benchmarks/test_metrics.csv'
if baseline.exists():pd.read_csv(baseline).to_csv(out/'fixed_financial_benchmarks.csv',index=False)
display(g.loc[g.Strategy.eq('Pretest Selected Gate') & g.Metric.isin(('annualized_return','sharpe_ratio','maximum_drawdown'))])
print('Saved five-seed tables:',out)


,Strategy,Metric,N,Mean,SD,Median,Min,Max
54,Pretest Selected Gate,annualized_return,5,0.193979,0.008253,0.193937,0.180694,0.201867
57,Pretest Selected Gate,maximum_drawdown,5,0.304546,0.002440,0.304953,0.301879,0.307632
58,Pretest Selected Gate,sharpe_ratio,5,1.169518,0.037719,1.173914,1.112952,1.209827


Saved five-seed tables: /content/drive/MyDrive/tw50_2004_2025_alpha_dynmoe/five_seed_continuous_2016_2025/summary
